In [1]:
import pandas as pd

DATA_PATH = "../data/raw/yellow_tripdata_2025-01.parquet"

df = pd.read_parquet(DATA_PATH)
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee            float64   

In [3]:
df["duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["duration"].describe()

count    3.475226e+06
mean     1.501812e+01
std      3.871358e+01
min     -5.147232e+04
25%      7.283333e+00
50%      1.170000e+01
75%      1.833333e+01
max      5.626317e+03
Name: duration, dtype: float64

In [4]:
print("Negative durations:", (df["duration"] < 0).sum())
print("Zero durations:", (df["duration"] == 0).sum())
print("Positive durations:", (df["duration"] > 0).sum())

Negative durations: 124
Zero durations: 1927
Positive durations: 3473175


In [5]:
print("Under 1 minute:", (df["duration"] < 1).sum())
print("Over 60 minutes:", (df["duration"] > 60).sum())
print(
    "Valid trips (1–60 min):",
    df["duration"].between(1, 60).sum()
)

Under 1 minute: 40111
Over 60 minutes: 31867
Valid trips (1–60 min): 3403248


In [6]:
long_trips = df[df["duration"] > 60]

long_trips[
    ["trip_distance", "duration"]
].describe()

,trip_distance,duration
count,31867.000000,31867.000000
mean,19.808342,119.495471
std,359.654254,242.532963
min,0.000000,60.016667
25%,12.200000,63.750000
50%,17.200000,69.083333
75%,19.290000,79.700000
max,44730.300000,5626.316667


In [7]:
zero_distance = df[df["trip_distance"] == 0]

print("Zero-distance trips:", len(zero_distance))

zero_distance["duration"].describe()

Zero-distance trips: 90893


count    90893.000000
mean        11.676019
std         27.820032
min          0.000000
25%          0.500000
50%         10.166667
75%         17.533333
max       4184.000000
Name: duration, dtype: float64

In [8]:
zero_distance[
    ["trip_distance", "duration", "fare_amount"]
].head(10)

,trip_distance,duration,fare_amount
92,0.0,0.000000,20.06
204,0.0,0.166667,12.00
358,0.0,0.133333,30.00
505,0.0,31.833333,50.50
619,0.0,0.083333,20.00
706,0.0,0.033333,3.00
1073,0.0,2.933333,5.00
1345,0.0,0.133333,20.00
1354,0.0,2.783333,3.00
1496,0.0,0.066667,85.00


In [9]:
df["trip_distance"].describe(
    percentiles=[0.90, 0.95, 0.99, 0.999]
)

count    3.475226e+06
mean     5.855126e+00
std      5.646016e+02
min      0.000000e+00
90%      7.700000e+00
95%      1.183000e+01
99%      1.950000e+01
99.9%    2.885775e+01
max      2.764236e+05
Name: trip_distance, dtype: float64

In [10]:
for threshold in [30, 50, 100, 1000]:
    count = (df["trip_distance"] > threshold).sum()
    print(f"Trips over {threshold} miles: {count}")

Trips over 30 miles: 2789
Trips over 50 miles: 533
Trips over 100 miles: 162
Trips over 1000 miles: 116


In [11]:
df.loc[
    df["trip_distance"] > 100,
    ["trip_distance", "fare_amount", "duration"]
].sort_values("trip_distance", ascending=False).head(20)

,trip_distance,fare_amount,duration
3275762,276423.57,-4.75,9.0
3188501,276099.95,9.13,7.0
3240337,222167.49,31.19,26.0
3112173,206137.99,24.89,24.0
3374854,202771.63,10.10,13.0
3081597,189687.43,12.70,18.0
3069039,181139.99,6.33,6.0
3312563,168079.57,-4.00,11.0
3195037,167452.94,-4.00,27.0
3138510,164959.95,14.05,15.0


In [12]:
valid_duration = df[df["duration"] > 0].copy()

valid_duration["speed_mph"] = (
    valid_duration["trip_distance"]
    / (valid_duration["duration"] / 60)
)

valid_duration["speed_mph"].describe(
    percentiles=[0.90, 0.95, 0.99, 0.999]
)

count    3.473175e+06
mean     2.573553e+01
std      3.873889e+03
min      0.000000e+00
90%      1.943662e+01
95%      2.506782e+01
99%      3.554229e+01
99.9%    4.727742e+01
max      3.722497e+06
Name: speed_mph, dtype: float64

In [13]:
for threshold in [50, 65, 80, 100]:
    count = (valid_duration["speed_mph"] > threshold).sum()
    print(f"Trips above {threshold} mph: {count}")

Trips above 50 mph: 2243
Trips above 65 mph: 1090
Trips above 80 mph: 982
Trips above 100 mph: 899


In [14]:
print("Missing pickup:", df["PULocationID"].isna().sum())
print("Missing dropoff:", df["DOLocationID"].isna().sum())

print("Zero pickup:", (df["PULocationID"] == 0).sum())
print("Zero dropoff:", (df["DOLocationID"] == 0).sum())

Missing pickup: 0
Missing dropoff: 0
Zero pickup: 0
Zero dropoff: 0


In [15]:
print("Negative pickup IDs:", (df["PULocationID"] < 0).sum())
print("Negative dropoff IDs:", (df["DOLocationID"] < 0).sum())

Negative pickup IDs: 0
Negative dropoff IDs: 0


In [17]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [18]:
from src.data_pipeline.transform import transform_data

df_clean = transform_data(df)

print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Removed rows:", len(df) - len(df_clean))

Original rows: 3475226
Cleaned rows: 3368988
Removed rows: 106238


In [ ]:
suspicious = df[df["trip_distance"] <= 0].copy()

print("Suspicious records:", len(suspicious))

print("Zero duration:",
      ((suspicious["duration"] <= 0) & (suspicious["fare_amount"] <= 0)).sum())

print("Positive duration:",
      (suspicious["duration"] > 0).sum())

print("Positive fare:",
      (suspicious["fare_amount"] > 0).sum())

print("Negative or zero fare:",
      (suspicious["fare_amount"] <= 0).sum())

Suspicious records: 90893
Zero duration: 45
Positive duration: 90201
Positive fare: 75846
Negative or zero fare: 15047
